# Task 6 — Run and Analyze the Judge

In [1]:
import json
import os
import sys
import time
from enum import Enum
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import JUDGE_MODEL, NEBIUS_BASE_URL, OUTPUT_EXCEL
from shared.constants import (
    CRITERIA,
    JUDGE_CRITERIA,
    JUDGE_SYSTEM_PROMPT,
    RUBRIC,
    final_score,
)
from shared.utils import rate_cost, rate_latency

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

df = pd.read_excel(OUTPUT_EXCEL)
print(f"{len(df)} products loaded from {OUTPUT_EXCEL}")

50 products loaded from /Users/nisim/Desktop/Nebuis Academy/Lesson-1-homeowrk/task_3/assignment_01.xlsx


In [2]:
class RatingEnum(str, Enum):
    good = "good"
    ok = "ok"
    bad = "bad"


class CriterionResult(BaseModel):
    explanation: str = Field(description="Reasoning for the verdict")
    verdict: RatingEnum = Field(description="Rating: good, ok, or bad")


class JudgeOutput(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult


def build_judge_user_message(row: pd.Series) -> str:
    return (
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}\n\n"
        f"Generated description:\n{row['generated_description']}"
    )


def judge_description(row: pd.Series, model: str = JUDGE_MODEL) -> JudgeOutput:
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": build_judge_user_message(row)},
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "judge_output",
                "strict": True,
                "schema": JudgeOutput.model_json_schema(),
            },
        },
    )
    return JudgeOutput.model_validate_json(response.choices[0].message.content)


def extract_verdicts(result: JudgeOutput) -> dict[str, str]:
    return {c: getattr(result, c.lower()).verdict.value for c in JUDGE_CRITERIA}

## 1. Sanity Check (5 products)

Run the judge on 5 products and manually review the explanations and verdicts.

In [3]:
sanity_indices = np.linspace(0, len(df) - 1, 5, dtype=int).tolist()

for i in sanity_indices:
    row = df.iloc[i]
    print(f"=== [{i}] {row['product_name']} ===")
    print(f"Description: {row['generated_description']}")
    print()

    result = judge_description(row)
    for c in JUDGE_CRITERIA:
        cr = getattr(result, c.lower())
        print(f"  {c}: {cr.verdict.value}")
        print(f"    {cr.explanation}")
    print()

=== [0] Apple iPhone 15 Pro ===
Description: Experience the power and performance of the Apple iPhone 15 Pro.  Fueled by the groundbreaking A17 Pro chip, this compact powerhouse delivers a seamless user experience with its 120Hz ProMotion display.  The durable titanium frame and Ceramic Shield glass ensure lasting protection, while USB‑C fast charging keeps you connected. Backed by a 1-year limited warranty, the iPhone 15 Pro is ready to elevate your mobile experience.

  Fluency: good
    The description reads naturally with smooth sentence flow, and the phrasing is clear and concise.
  Grammar: good
    There are no spelling or punctuation errors in the description.
  Tone: good
    The tone is friendly, persuasive, and professionally credible, maintaining a consistent sales voice throughout the description.
  Length: good
    The description contains 76 words, which falls within the 50-90 word range.
  Grounding: good
    The description accurately reflects the provided product info

## 2. Full Run

Run the judge on all products, compute `final_score`, and save back to the spreadsheet.

In [4]:
judge_results = []

for idx, row in df.iterrows():
    result = judge_description(row)
    verdicts = extract_verdicts(result)
    judge_results.append(verdicts)
    print(f"[{idx + 1}/{len(df)}] {row['product_name']}: "
          f"{', '.join(f'{c}={v}' for c, v in verdicts.items())}")

print(f"\nJudged {len(judge_results)} products.")

[1/50] Apple iPhone 15 Pro: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[2/50] Samsung Galaxy S24 Ultra: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[3/50] Google Pixel 8 Pro: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[4/50] Sony WH‑1000XM5 Headphones: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=ok
[5/50] Bose QuietComfort Ultra Earbuds: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[6/50] Amazon Echo Dot (5th Gen): Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[7/50] Dell XPS 13 9310 Laptop: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[8/50] Apple MacBook Air 13″ (M3): Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[9/50] Microsoft Surface Pro 10: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[10/50] Garmin Forerunner 255: Fluency=good, Grammar=good, Tone=good, Length=good, Grounding=good
[11/

In [5]:
# Write judge verdicts + programmatic ratings into the DataFrame
for idx, verdicts in enumerate(judge_results):
    for c, v in verdicts.items():
        df.at[idx, f"judge_{c}"] = v

    full_ratings = {
        **verdicts,
        "Latency": rate_latency(df.at[idx, "latency_ms"]),
        "Cost": rate_cost(df.at[idx, "cost_usd"] if "cost_usd" in df.columns else 0),
    }
    df.at[idx, "judge_final_score"] = final_score(full_ratings)

print("Pass/fail distribution (judge):")
print(df["judge_final_score"].value_counts())

df.to_excel(OUTPUT_EXCEL, index=False)
print(f"\nSaved to {OUTPUT_EXCEL}")

Pass/fail distribution (judge):
judge_final_score
pass    46
fail     4
Name: count, dtype: int64

Saved to /Users/nisim/Desktop/Nebuis Academy/Lesson-1-homeowrk/task_3/assignment_01.xlsx


## 3. Compare to Human Evaluation

For each criterion, compute the agreement rate between the judge's verdicts and the manual scores from Task 3.

In [6]:
# Find rows that have both human and judge ratings
human_rated = df[df["final_score"].isin(["pass", "fail"])].copy()
print(f"{len(human_rated)} products have both human and judge ratings.")
print()

if len(human_rated) > 0:
    print(f"{'Criterion':<12} {'Agree':>6} {'Total':>6} {'Rate':>8}")
    print("-" * 36)
    for c in JUDGE_CRITERIA:
        human_col = human_rated[c].astype(str)
        judge_col = human_rated[f"judge_{c}"].astype(str)
        agree = (human_col == judge_col).sum()
        total = len(human_rated)
        print(f"{c:<12} {agree:>6} {total:>6} {agree/total:>7.0%}")

    # Overall final_score agreement
    fs_agree = (human_rated["final_score"] == human_rated["judge_final_score"]).sum()
    print(f"\nFinal score agreement: {fs_agree}/{len(human_rated)} ({fs_agree/len(human_rated):.0%})")
else:
    print("No human-rated products found. Run Task 3 first.")

15 products have both human and judge ratings.

Criterion     Agree  Total     Rate
------------------------------------
Fluency          15     15    100%
Grammar          15     15    100%
Tone             14     15     93%
Length           14     15     93%
Grounding        11     15     73%

Final score agreement: 14/15 (93%)


## 4. Criterion-by-Criterion Judging

Instead of evaluating all 5 criteria in a single call, we run the judge separately for each criterion (one call per criterion per product). This isolates each evaluation and prevents criteria from influencing each other.

**Why this might change results:** When evaluating all criteria at once, the model may anchor on early judgments — e.g., if it rates Fluency as "bad", it might be biased toward harsher ratings on subsequent criteria. Isolating each criterion removes this cross-contamination.

In [7]:
class SingleCriterionOutput(BaseModel):
    explanation: str = Field(description="Reasoning for the verdict")
    verdict: RatingEnum = Field(description="Rating: good, ok, or bad")


def build_single_criterion_prompt(criterion: str) -> str:
    defs = RUBRIC[criterion]
    return (
        f"You are an evaluation judge for e-commerce product descriptions.\n\n"
        f"Evaluate the description on a single criterion: {criterion}.\n\n"
        f"Rubric for {criterion}:\n"
        f"  good: {defs['good']}\n"
        f"  ok: {defs['ok']}\n"
        f"  bad: {defs['bad']}\n\n"
        f"You will receive the original product information and the generated description.\n"
        f"First explain your reasoning, then give your verdict."
    )


def judge_single_criterion(row: pd.Series, criterion: str) -> str:
    messages = [
        {"role": "system", "content": build_single_criterion_prompt(criterion)},
        {"role": "user", "content": build_judge_user_message(row)},
    ]
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=messages,
        temperature=0.0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "single_criterion_output",
                "strict": True,
                "schema": SingleCriterionOutput.model_json_schema(),
            },
        },
    )
    parsed = SingleCriterionOutput.model_validate_json(response.choices[0].message.content)
    return parsed.verdict.value

In [8]:
isolated_results: list[dict[str, str]] = []

for idx, row in df.iterrows():
    verdicts = {}
    for c in JUDGE_CRITERIA:
        verdicts[c] = judge_single_criterion(row, c)
    isolated_results.append(verdicts)
    print(f"[{idx + 1}/{len(df)}] {row['product_name']}: "
          f"{', '.join(f'{c}={v}' for c, v in verdicts.items())}")

print(f"\nDone. Judged {len(isolated_results)} products criterion-by-criterion.")

[1/50] Apple iPhone 15 Pro: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=good
[2/50] Samsung Galaxy S24 Ultra: Fluency=good, Grammar=ok, Tone=good, Length=good, Grounding=ok
[3/50] Google Pixel 8 Pro: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[4/50] Sony WH‑1000XM5 Headphones: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[5/50] Bose QuietComfort Ultra Earbuds: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[6/50] Amazon Echo Dot (5th Gen): Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[7/50] Dell XPS 13 9310 Laptop: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[8/50] Apple MacBook Air 13″ (M3): Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[9/50] Microsoft Surface Pro 10: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=ok
[10/50] Garmin Forerunner 255: Fluency=good, Grammar=ok, Tone=good, Length=ok, Grounding=good
[11/50] Fitbit Charge 6: Fluency=good, Grammar=ok, Tone=

In [9]:
# Write isolated verdicts to DataFrame
for idx, verdicts in enumerate(isolated_results):
    for c, v in verdicts.items():
        df.at[idx, f"isolated_{c}"] = v

# Compare isolated vs. combined judge vs. human
if len(human_rated) > 0:
    print(f"{'Criterion':<12} {'Combined':>10} {'Isolated':>10}  (agreement with human)")
    print("-" * 50)
    hr_idx = human_rated.index
    for c in JUDGE_CRITERIA:
        human_vals = df.loc[hr_idx, c].astype(str)
        combined_vals = df.loc[hr_idx, f"judge_{c}"].astype(str)
        isolated_vals = df.loc[hr_idx, f"isolated_{c}"].astype(str)
        combined_agree = (human_vals == combined_vals).mean()
        isolated_agree = (human_vals == isolated_vals).mean()
        print(f"{c:<12} {combined_agree:>9.0%} {isolated_agree:>9.0%}")

# Compare combined vs. isolated (all products)
print("\nCombined vs. Isolated agreement (all products):")
for c in JUDGE_CRITERIA:
    agree = (df[f"judge_{c}"] == df[f"isolated_{c}"]).mean()
    print(f"  {c}: {agree:.0%}")

df.to_excel(OUTPUT_EXCEL, index=False)
print(f"\nSaved to {OUTPUT_EXCEL}")

Criterion      Combined   Isolated  (agreement with human)
--------------------------------------------------
Fluency           100%      100%
Grammar           100%        0%
Tone               93%       93%
Length             93%        0%
Grounding          73%       40%

Combined vs. Isolated agreement (all products):
  Fluency: 96%
  Grammar: 10%
  Tone: 96%
  Length: 10%
  Grounding: 36%

Saved to /Users/nisim/Desktop/Nebuis Academy/Lesson-1-homeowrk/task_3/assignment_01.xlsx


## 5. Analysis

### 5a. Trade-offs: Human Evaluation vs. LLM-as-a-Judge

| Dimension | Human Evaluation | LLM-as-a-Judge |
|-----------|-----------------|----------------|
| **Cost** | Expensive — requires paid annotator time, ~minutes per product | Cheap — a few cents per 1K products at current API pricing |
| **Scale** | Slow — practical for tens of items, not thousands | Fast — can evaluate thousands in minutes |
| **Consistency** | Variable — annotators drift over time, disagree with each other, or get fatigued | Highly consistent — same prompt produces the same logic every time |
| **Accuracy** | Strong on subjective criteria (tone, fluency) where humans understand nuance | Good on objective criteria (length, grammar), weaker on subtle grounding issues or cultural tone |
| **Bias** | Annotator-specific biases (leniency, anchoring on previous ratings) | Model-specific biases (positivity bias, anchoring across criteria in a single call) |
| **Adaptability** | Can handle edge cases, novel product categories, and ambiguous situations | Bound by the rubric in the prompt — misses anything not explicitly specified |
